<a href="https://colab.research.google.com/github/Gianbattistabsn/FAIML-RL-26/blob/alessandro-PPO-SAC/part2/clone_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if not IN_COLAB:
    raise RuntimeError("Questo setup è pensato per Colab")

from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/Gianbattistabsn/FAIML-RL-26.git"
REPO_BRANCH = "alessandro-PPO-SAC"
REPO_ROOT = "/content/FAIML-RL-26"
VENV = "/content/rl_env"

# ============================================================
# Fix missing venv support in Colab
# ============================================================
print("Installing python venv support...")
subprocess.run(
    ["apt-get", "update"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

subprocess.run(
    ["apt-get", "install", "-y", "python3.12-venv"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# ============================================================
# Clone repo
# ============================================================
if not os.path.exists(REPO_ROOT):
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_ROOT],
        check=True
    )

# ============================================================
# Recreate clean virtualenv
# ============================================================
if os.path.exists(VENV):
    subprocess.run(["rm", "-rf", VENV], check=True)

print("Creating virtual environment...")
subprocess.run(
    [sys.executable, "-m", "venv", VENV],
    check=True
)

PY = f"{VENV}/bin/python"
PIP = f"{VENV}/bin/pip"

# ============================================================
# Upgrade tooling
# ============================================================
print("Upgrading pip...")
subprocess.run([PIP, "install", "--upgrade", "pip", "setuptools", "wheel"], check=True)

# ============================================================
# Install compatible stack
# ============================================================
print("Installing RL dependencies...")

subprocess.run([
    PIP, "install",
    "numpy==1.26.4",
    "gym==0.26.2",
    "tensorboard==2.16.2",
    "stable-baselines3==2.3.2",
    "pybullet",
    "opencv-python-headless<4.10"
], check=True)

# Repo requirements without dependency resolver chaos
subprocess.run([
    PIP, "install",
    "-r", f"{REPO_ROOT}/requirements.txt",
    "--no-deps"
], check=False)

# Local panda-gym
subprocess.run([
    PIP, "install",
    "-e", f"{REPO_ROOT}/part2/panda-gym"
], check=True)

# Optional display tools
subprocess.run([
    PIP, "install",
    "pyvirtualdisplay"
], check=False)

# ============================================================
# Verify
# ============================================================
print("\n=== VERIFY ===")

subprocess.run([
    PY,
    "-c",
    """
import numpy as np
import gym
import torch
print('NumPy:', np.__version__)
print('Gym:', gym.__version__)
print('Torch OK')
print('All imports OK')
"""
], check=True)

print("\nSetup complete.")
print("Interpreter:")
print(PY)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Installing python venv support...
Creating virtual environment...
Upgrading pip...
Installing RL dependencies...

=== VERIFY ===

Setup complete.
Interpreter:
/content/rl_env/bin/python


In [3]:
import sys
import os

REPO_ROOT = "/content/FAIML-RL-26"
PART2_DIR = os.path.join(REPO_ROOT, "part2")
LOCAL_PANDA_GYM = os.path.join(PART2_DIR, "panda-gym")

for p in [PART2_DIR, LOCAL_PANDA_GYM]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(sys.path[:3])

['/content/FAIML-RL-26/part2/panda-gym', '/content/FAIML-RL-26/part2', '/content']


In [4]:

# --- Sanity check: verify all key imports work ---
import gymnasium as gym
import numpy as np
import torch
import panda_gym  # registers PandaPush-v3 etc.
# from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback
from rand_wrapper import RandomizationWrapper

print("All imports OK")
print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
print(f"Registered panda envs: {[e for e in gym.envs.registry if 'Panda' in e][:5]}")


All imports OK
torch 2.11.0+cu130 | CUDA available: False
Registered panda envs: ['PandaReach-v3', 'PandaReachJoints-v3', 'PandaReachDense-v3', 'PandaReachJointsDense-v3', 'PandaSlide-v3']


## Training Configuration

Edit the variables below, then run the Training cell.


In [5]:

# ── Adjust these before running ──────────────────────────────────────────────
SAMPLING_STRATEGY = "none"    # "none" | "udr" | "adr"
ENV_TYPE          = "source"  # "source" | "target"
TIMESTEPS         = 200_000   # total training steps
LOAD_MODEL        = False     # True → load existing zip instead of training
# ─────────────────────────────────────────────────────────────────────────────


## Train SAC on PandaPush-v3


In [ ]:

import os

save_name = os.path.join(
    REPO_ROOT, "part2", "models",
    f"sac_push_{SAMPLING_STRATEGY}_{ENV_TYPE}_{TIMESTEPS // 1000}k"
)
os.makedirs(os.path.dirname(save_name), exist_ok=True)

env = gym.make("PandaPush-v3", render_mode="rgb_array", type=ENV_TYPE, reward_type="dense")
if SAMPLING_STRATEGY != "none":
    env = RandomizationWrapper(env, mode=SAMPLING_STRATEGY)

if LOAD_MODEL:
    model = SAC.load(f"{save_name}.zip")
    model.set_env(DummyVecEnv([lambda: env]))
    print(f"Model loaded from {save_name}.zip")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Training on device: {device}")

    vec_env = DummyVecEnv([lambda: env])

    model = SAC(
        policy="MultiInputPolicy",
        env=vec_env,
        device=device,
        verbose=1,
        learning_rate=1e-3,
        buffer_size=200_000,
        batch_size=256,
        tensorboard_log=f"{save_name}/logs",
    )

    checkpoint_cb = CheckpointCallback(
        save_freq=50_000,
        save_path=f"{save_name}/checkpoints",
        name_prefix="model",
    )

    model.learn(total_timesteps=TIMESTEPS, callback=checkpoint_cb, progress_bar=True)
    model.save(save_name)
    print(f"Model saved to {save_name}.zip")


Created object with mass: 1.0
Training on device: cpu
Using cpu device
Logging to /content/FAIML-RL-26/part2/models/sac_push_none_source_200k/logs/SAC_1


Output()

---------------------------------
| rollout/           |          |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 4        |
|    fps             | 42       |
|    time_elapsed    | 4        |
|    total_timesteps | 200      |
| train/             |          |
|    actor_loss      | -4.25    |
|    critic_loss     | 0.0717   |
|    ent_coef        | 0.907    |
|    ent_coef_loss   | -0.498   |
|    learning_rate   | 0.001    |
|    n_updates       | 99       |
---------------------------------
---------------------------------
| rollout/           |          |
|    success_rate    | 0.125    |
| time/              |          |
|    episodes        | 8        |
|    fps             | 34       |
|    time_elapsed    | 10       |
|    total_timesteps | 351      |
| train/             |          |
|    actor_loss      | -4.64    |
|    critic_loss     | 0.0274   |
|    ent_coef        | 0.779    |
|    ent_coef_loss   | -1.25    |
|    learning_

## Evaluate the Trained Model


In [ ]:

N_EVAL_EPISODES = 20

eval_env = gym.make("PandaPush-v3", render_mode="rgb_array", type=ENV_TYPE, reward_type="dense")

episode_returns = []
successes = []

for ep in range(1, N_EVAL_EPISODES + 1):
    obs, info = eval_env.reset()
    done = False
    ep_return = 0.0

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        ep_return += float(reward)
        done = terminated or truncated

    episode_returns.append(ep_return)
    if isinstance(info, dict) and "is_success" in info:
        successes.append(float(info["is_success"]))
    print(f"Episode {ep:03d} | return = {ep_return:.3f}")

eval_env.close()

returns = np.array(episode_returns)
print(f"\n=== Eval over {N_EVAL_EPISODES} episodes ===")
print(f"Mean return : {returns.mean():.3f} ± {returns.std():.3f}")
print(f"Min / Max   : {returns.min():.3f} / {returns.max():.3f}")
if successes:
    print(f"Success rate: {np.mean(successes):.2%}")
